# Week 3 · Module 1 Practical — Build an Agent From Atoms 🤖

**PMS RoBoTics Research Center · 4-Week Internship & Training Program**
**Week 3 · Module 1: What are AI agents and how they differ from chatbots**

---

### The ladder you'll climb (no frameworks — every line is yours)

| # | Rung | What you build |
|---|------|----------------|
| 0 | Plain call | Watch it fail at anything real-time |
| 1 | Structured output | Text becomes a dict Python can branch on |
| 2 | Tools | 3 real functions + a registry (a dict!) |
| 3 | The decision | Hand-rolled function calling — one round trip |
| 4 | **The loop** | 15 lines → THINK / ACT / OBSERVE trace, live |
| 5 | Break & defend | 4 failure modes, 4 one-line fixes |

> 🧭 Keep slide 8 (sequence diagram) and slide 9 (state diagram) open — the code prints the SAME states and steps you saw drawn.

## Step 0 — Setup 🔑

In [ ]:
%pip install -q -U openai

In [ ]:
import json
from getpass import getpass
from openai import OpenAI

API_KEY = getpass("Paste the OpenAI API key: ")
client = OpenAI(api_key=API_KEY)
MODEL = "gpt-4o-mini"

def ask(prompt, temperature=0.0, max_tokens=400):
    r = client.chat.completions.create(model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content.strip()

# SmartMart's "live" systems (in a real store these would be DB/API calls)
INVENTORY = {"P-101": 42, "P-102": 18, "P-103": 65, "P-104": 7, "P-105": 23,
             "P-118": 3, "P-119": 14, "P-120": 73}
PRICES    = {"P-101": 1299, "P-102": 1699, "P-103": 899, "P-104": 2499,
             "P-105": 3499, "P-118": 12999, "P-119": 2799, "P-120": 599}
NAMES     = {"P-101": "Prima Electric Kettle 1.5L", "P-102": "Prima Electric Kettle 2.0L",
             "P-103": "Zenith Steel Kettle 1.0L",   "P-104": "Zenith Pro Kettle 1.8L",
             "P-105": "SwiftMix Mixer Grinder 750W","P-118": "CleanSweep Robot Vacuum",
             "P-119": "FreshBrew Coffee Maker 600ml","P-120": "ChopMaster Vegetable Chopper"}
ORDERS    = {"4521": {"status": "shipped", "eta": "tomorrow", "item": "P-105"},
             "7788": {"status": "processing", "eta": "3 days", "item": "P-101"}}

print("✅ Ready. SmartMart systems online:", len(INVENTORY), "products,", len(ORDERS), "orders")

---
## Rung 0 — The plain call fails 🙅

Ask a question that needs *live* data:

In [ ]:
print(ask("Is the Prima Electric Kettle 1.5L in stock at SmartMart right now? Exact count please.",
          temperature=0.3))

It either refuses ("no access to real-time data") or — worse — invents a number. The model **has no hands**: text in, text out is its entire interface. The data sits 3 cells up in `INVENTORY`, unreachable.

**The reframe that changes everything:** text out is not nothing. If the text is *structured*, our code can read it and act on the model's behalf.

---
## Rung 1 — Structured output: text becomes data 🧱

You've done this twice (W1D2 extractor, W2D4 judges). Today it gets a new job — expressing **intent**:

In [ ]:
INTENT = """Extract the customer's request. Return ONLY JSON (no fences):
{{"product_hint": "<words describing the product>", "wants": "price" | "stock" | "order_status"}}

Message: "{msg}"
"""

for msg in ["how much is the 750W mixer?",
            "kitna stock hai robot vacuum ka?",
            "where is my order 4521??"]:
    d = json.loads(ask(INTENT.format(msg=msg)))
    print(f"{msg:<38} → {d}")

In [ ]:
# The bridge moment: the model's words are now a dict our code can BRANCH on
d = json.loads(ask(INTENT.format(msg="how much is the 750W mixer?")))
if d["wants"] == "price":
    print("Python just made a decision based on what the LLM understood. ←—— the seed of agency")

---
## Rung 2 — Tools: functions with two audiences 🔧

The **body** runs for Python. The **docstring** is read by the LLM. Both matter:

In [ ]:
def check_inventory(product_id: str) -> str:
    """Returns how many units of a product are in stock right now.
    Args: product_id — catalog ID like 'P-101'."""
    if product_id not in INVENTORY:
        return f"Unknown product ID: {product_id}"
    return f"{INVENTORY[product_id]} units of {NAMES[product_id]} in stock"

def get_price(product_id: str) -> str:
    """Returns the current price in rupees of a product.
    Args: product_id — catalog ID like 'P-101'."""
    if product_id not in PRICES:
        return f"Unknown product ID: {product_id}"
    return f"{NAMES[product_id]} costs Rs. {PRICES[product_id]:,}"

def track_order(order_id: str) -> str:
    """Returns the status and delivery ETA of a customer order.
    Args: order_id — the order number like '4521'."""
    o = ORDERS.get(order_id)
    if not o:
        return f"No order found with ID {order_id}"
    return f"Order {order_id} ({NAMES[o['item']]}): {o['status']}, ETA {o['eta']}"

def list_products(keyword: str) -> str:
    """Searches the catalog by keyword; returns matching IDs, names and prices.
    Args: keyword — e.g. 'kettle'."""
    hits = [f"{pid}: {name} (Rs. {PRICES[pid]:,})" for pid, name in NAMES.items()
            if keyword.lower() in name.lower()]
    return "\n".join(hits) if hits else f"No products matching '{keyword}'"

# The registry — the entire 'plugin system' is a dict:
TOOLS = {f.__name__: f for f in [check_inventory, get_price, track_order, list_products]}
TOOL_DESCS = "\n".join(f"- {name}: {f.__doc__.strip()}" for name, f in TOOLS.items())
print(TOOL_DESCS)

In [ ]:
# Tools are just Python — prove it:
print(check_inventory("P-118"))
print(track_order("4521"))
print(list_products("kettle"))

**✏️ Your turn:** call `TOOLS["get_price"]("P-103")` — dictionary lookup + call. That dispatch move is exactly what the agent will do.

---
## Rung 3 — The decision: hand-rolled function calling 🎯

The prompt below IS the mechanism. Two allowed outputs: a **tool request** or a **final answer**. The model *writes*; our code *executes* (slide 8, steps 2–5):

In [ ]:
DECIDE = """You are SmartBot's reasoning engine. You can use these tools:
{tool_descriptions}

To use a tool, return ONLY this JSON (no other text, no fences):
  {{"tool": "<tool_name>", "args": {{"<arg>": "<value>"}}}}
If you already have enough information to answer, return ONLY:
  {{"answer": "<final answer for the customer>"}}

Conversation so far:
{transcript}"""

question = "Is the robot vacuum in stock?"

# THINK — one LLM call
decision = json.loads(ask(DECIDE.format(tool_descriptions=TOOL_DESCS, transcript=question)))
print("THINK   →", decision)

# ACT — our code executes the model's request
result = TOOLS[decision["tool"]](**decision["args"])
print("ACT     →", result)

# OBSERVE + second THINK — feed the result back
transcript = question + f"\n[observation from {decision['tool']}: {result}]"
final = json.loads(ask(DECIDE.format(tool_descriptions=TOOL_DESCS, transcript=transcript)))
print("THINK   →", final)
print()
print("🛒", final["answer"])

**You just executed slide 8, line by line:** question → LLM decides (step 3) → our code runs the tool (step 4) → observation back to LLM (step 6) → final answer (steps 7–8). Two LLM calls, one tool call, zero magic.

**✏️ Your turn:** change the question to "where's order 7788?" and re-run. Different tool, same machinery — *you didn't change any code*.

---
## Rung 4 — THE LOOP 🔁

Wrap rung 3 in `for step in range(max_steps)` — that's the whole difference between one trick and an agent. The trace prints slide 9's states live:

In [ ]:
def agent(question, max_steps=5, verbose=True):
    transcript = question
    for step in range(1, max_steps + 1):
        # ---- THINK ----
        decision = json.loads(ask(DECIDE.format(
            tool_descriptions=TOOL_DESCS, transcript=transcript)))
        if "answer" in decision:
            if verbose: print(f"  step {step} · THINK   → DONE")
            return decision["answer"]
        if verbose: print(f"  step {step} · THINK   → use {decision['tool']}({decision['args']})")
        # ---- ACT ----
        result = TOOLS[decision["tool"]](**decision["args"])
        if verbose: print(f"  step {step} · ACT     → {result}")
        # ---- OBSERVE ----
        transcript += f"\n[observation from {decision['tool']}: {result}]"
        if verbose: print(f"  step {step} · OBSERVE → appended to transcript")
    return "I couldn't finish within my step limit — let me connect you with our team."

print("🛒", agent("Is the robot vacuum in stock, and what does it cost?"))

In [ ]:
# The real test — a task that needs PLANNING nobody scripted:
print("🛒", agent("Which kettle is the cheapest, is it in stock, and what would THREE of them cost?"))

**Read the trace slowly.** The model had to: search kettles → compare prices → check stock of the winner → multiply by three → answer. **Nobody wrote that sequence.** The loop discovered it, one THINK at a time, each decision informed by the last OBSERVE.

That emergent planning is the entire difference between Week 2's librarian and Week 3's assistant.

**✏️ Your turn:** ask something impossible ("cancel my order 4521") — watch it either use track_order and explain honestly, or answer that it lacks a cancellation tool. Both are correct behaviour: it can only act through the tools YOU registered.

---
## Rung 5 — Break it, then defend it 🛡️

Four failure modes from slide 12. First, cause them; then install the defended loop:

In [ ]:
# Break 1: malformed JSON — force it by confusing the format
try:
    bad = ask("Reply with the words: I think the tool is check_inventory maybe?")
    json.loads(bad)
except json.JSONDecodeError as e:
    print("💥 Break 1 (malformed JSON):", e)

# Break 2: unknown tool — simulate the model asking for a tool we never registered
decision = {"tool": "cancel_order", "args": {"order_id": "4521"}}
try:
    TOOLS[decision["tool"]](**decision["args"])
except KeyError:
    print("💥 Break 2 (unknown tool): KeyError — the registry has no 'cancel_order'")

# Break 3: tool raises — bad arguments
try:
    check_inventory()          # missing required arg
except TypeError as e:
    print("💥 Break 3 (tool raises):", e)

print("\nBreak 4 (infinite loop) is prevented, not caught — max_steps IS the defense.")

In [ ]:
def robust_agent(question, max_steps=5, verbose=True):
    """The defended loop: every failure becomes an OBSERVATION and the loop continues."""
    transcript = question
    for step in range(1, max_steps + 1):
        # THINK — defend against malformed JSON
        raw = ask(DECIDE.format(tool_descriptions=TOOL_DESCS, transcript=transcript))
        try:
            decision = json.loads(raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip())
        except json.JSONDecodeError:
            transcript += "\n[system: your last reply was not valid JSON — reply with ONLY the JSON format]"
            if verbose: print(f"  step {step} · THINK   → ⚠️ bad JSON, asked to retry")
            continue
        if "answer" in decision:
            if verbose: print(f"  step {step} · THINK   → DONE")
            return decision["answer"]
        tool, args = decision.get("tool"), decision.get("args", {})
        # ACT — defend against unknown tools and tool crashes
        if tool not in TOOLS:
            result = f"Unknown tool '{tool}'. Available: {', '.join(TOOLS)}"
        else:
            try:
                result = TOOLS[tool](**args)
            except Exception as e:
                result = f"Tool error: {type(e).__name__}: {e}"
        if verbose: print(f"  step {step} · THINK   → use {tool}({args})")
        if verbose: print(f"  step {step} · ACT     → {result}")
        # OBSERVE
        transcript += f"\n[observation from {tool}: {result}]"
    return "I couldn't finish within my step limit — let me connect you with our team."

# The nastiest test we have: a typo'd order + a real request in one message
print("🛒", robust_agent("track order 9999 and also tell me the price of the coffee maker"))

**The pattern in one sentence:** *errors become observations*. The loop that creates the risk also absorbs it — a failed tool call is just more information for the next THINK.

**✏️ Your turn:** register a deliberately broken tool (`def flaky(x): raise RuntimeError("db down")`, add to `TOOLS`, rebuild `TOOL_DESCS`) and watch `robust_agent` route around it gracefully.

---
## 🏁 Finale — AgentV0, the class

In [ ]:
class AgentV0:
    """Hand-rolled SmartMart agent. Zero frameworks — every line is Week 3 Day 1."""

    def __init__(self, tools=None, max_steps=5):
        self.tools = tools or TOOLS
        self.tool_descs = "\n".join(f"- {n}: {f.__doc__.strip()}" for n, f in self.tools.items())
        self.max_steps = max_steps
        self.trace = []

    def run(self, question):
        self.trace = []
        transcript = question
        for step in range(1, self.max_steps + 1):
            raw = ask(DECIDE.format(tool_descriptions=self.tool_descs, transcript=transcript))
            try:
                d = json.loads(raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip())
            except json.JSONDecodeError:
                transcript += "\n[system: reply with ONLY valid JSON]"
                self.trace.append((step, "THINK", "bad JSON — retry"))
                continue
            if "answer" in d:
                self.trace.append((step, "DONE", d["answer"]))
                return d["answer"]
            tool, args = d.get("tool"), d.get("args", {})
            result = (self.tools[tool](**args) if tool in self.tools
                      else f"Unknown tool '{tool}'")
            self.trace.append((step, "THINK→ACT", f"{tool}({args}) → {result}"))
            transcript += f"\n[observation from {tool}: {result}]"
        return "Step limit reached."

    def show_trace(self):
        for step, state, info in self.trace:
            print(f"  {step} · {state:<9} {info[:90]}")

bot = AgentV0()
print("🛒", bot.run("what's cheaper — the chopper or the budget kettle — and are both in stock?"))
print("\n--- trace (compare with slide 9's state diagram) ---")
bot.show_trace()

In [ ]:
# 💬 Talk to your agent — 'quit' to stop, 't' to see the last trace.
while True:
    user_msg = input("You: ")
    if user_msg.strip().lower() in ("quit", "exit", "q"):
        print("👋 Tomorrow: workflow patterns — controlled pipelines built from these same atoms.")
        break
    if user_msg.strip().lower() == "t":
        bot.show_trace(); continue
    print("🛒", bot.run(user_msg))

---
## 🏆 Homework (20 min, feeds Friday's agent project)

1. **Add a tool** — `apply_discount(product_id)`: returns the price after the current 10% kitchen offer (only for kitchen products — P-101..P-105, P-119, P-120; otherwise say the offer doesn't apply). Register it, rebuild `TOOL_DESCS`, and ask: *"what would the air fryer cost with the discount?"* — wait, P-107 isn't in this notebook's INVENTORY… what does your agent do? (That behaviour is your answer to write down.)
2. **Trace on paper** — run `bot.run("is the cheapest kettle in stock?")`, then draw the run as slide 9's state diagram: one arrow per transition, labelled with the tool used. Photo it — Friday's project asks for the same artifact.
3. **Break the docstring** — change `check_inventory`'s docstring to just "checks things" and re-run the multi-step task 3 times. How often does the agent pick the wrong tool now? One sentence: why the docstring is prompt engineering.

### What's next
**Module 2 — Workflow patterns:** before agents run free, the controlled shapes — prompt chaining, routing, parallelization, orchestrator-workers — each built in ~20 lines on top of today's atoms, each with its own diagram. When do you NEED a loop, and when is a pipeline safer, cheaper, and just better?

---
*PMS RoBoTics Research Center · pmsroboticsrc@gmail.com · (+91) 9960664674*